In [37]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# VADER Sentiment Analysis
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# NLTK for basic NLP preprocessing
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

# Download required NLTK data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('wordnet')

# Setup stopwords
stop_words = set(stopwords.words('english'))

# Initialize VADER analyzer
analyzer = SentimentIntensityAnalyzer()


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [7]:
!pip install vaderSentiment

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 5.3 MB/s eta 0:00:00


In [18]:
# prompt: read the csv file

df = pd.read_csv('/content/amzn_customer_reviews.csv')
df.head()

,Pseudo,Title,Review,Rating,Verified Purchase,Date
0,Assault Kittens,\nReally good option for portability.\n,\nThis laptop is very impressive for its form ...,4.5 out of 5 stars,Verified Purchase,"Reviewed in the United States on June 18, 2021"
1,Kenneth Cramer,\nExcellent portable gaming laptop\n,\nI'm writing this review for anyone who's on ...,5.0 out of 5 stars,Verified Purchase,"Reviewed in the United States on July 7, 2021"
2,Assault Kittens,\nBEST 14inch Laptop in the World\n,"\nI sold my M1 MacBook for this, and it was on...",1.0 out of 5 stars,Verified Purchase,"Reviewed in the United States on June 18, 2021"
3,Assault Kittens,\nWas great till it wasn't\n,\nThis laptop started off excellent. Screen lo...,5.0 out of 5 stars,Verified Purchase,"Reviewed in the United States on July 7, 2021"
4,Tanmay,\nsolid laptop\n,\npros:1) keyboard: razer laptops are still th...,5.0 out of 5 stars,Verified Purchase,"Reviewed in the United States on July 1, 2021"


In [11]:
# prompt: return the first 3 rowa

df.head(3)

,Pseudo,Title,Review,Rating,Verified Purchase,Date
0,Assault Kittens,\nReally good option for portability.\n,\nThis laptop is very impressive for its form ...,4.5 out of 5 stars,Verified Purchase,"Reviewed in the United States on June 18, 2021"
1,Kenneth Cramer,\nExcellent portable gaming laptop\n,\nI'm writing this review for anyone who's on ...,5.0 out of 5 stars,Verified Purchase,"Reviewed in the United States on July 7, 2021"
2,Assault Kittens,\nBEST 14inch Laptop in the World\n,"\nI sold my M1 MacBook for this, and it was on...",1.0 out of 5 stars,Verified Purchase,"Reviewed in the United States on June 18, 2021"


In [22]:

import re


# Helper cleaning function
def clean_text(text):
    if pd.isnull(text):
        return ""

    # 1. Remove leading/trailing spaces
    text = text.strip()

    # 2. Remove extra internal whitespace
    text = re.sub(r'\s+', ' ', text)

    # 3. Lowercase
    text = text.lower()
    # 5. Remove stopwords (optional)
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word not in stop_words]

    return ' '.join(tokens)

# Apply cleaning to both columns
df['Review'] = df['Review'].apply(clean_text)
df['Title'] = df['Title'].apply(clean_text)


In [23]:
df.head(3)

,Pseudo,Title,Review,Rating,Verified Purchase,Date
0,Assault Kittens,really good option portability,laptop impressive form factor really good bala...,4.5 out of 5 stars,Verified Purchase,"Reviewed in the United States on June 18, 2021"
1,Kenneth Cramer,excellent portable gaming laptop,im writing review anyone whos fence purchasing...,5.0 out of 5 stars,Verified Purchase,"Reviewed in the United States on July 7, 2021"
2,Assault Kittens,best 14inch laptop world,sold m1 macbook one best decisions lifeif youv...,1.0 out of 5 stars,Verified Purchase,"Reviewed in the United States on June 18, 2021"


In [24]:
vader = SentimentIntensityAnalyzer()
df["Score"] = df["Review"].apply(
    lambda review: vader.polarity_scores(review)
)
df["Compound"] = df["Score"].apply(
    lambda score_dict: score_dict["compound"]
)
df["Sentiment"] = df["Compound"].apply(
    lambda sent: "positive" if sent > 0 else ("neutral" if sent == 0 else "negative")
)
df["Sentiment"].value_counts()

,count
Sentiment,
positive,943
negative,108
neutral,85


In [25]:
df.head(3)

,Pseudo,Title,Review,Rating,Verified Purchase,Date,Score,Compound,Sentiment
0,Assault Kittens,really good option portability,laptop impressive form factor really good bala...,4.5 out of 5 stars,Verified Purchase,"Reviewed in the United States on June 18, 2021","{'neg': 0.033, 'neu': 0.606, 'pos': 0.361, 'co...",0.9891,positive
1,Kenneth Cramer,excellent portable gaming laptop,im writing review anyone whos fence purchasing...,5.0 out of 5 stars,Verified Purchase,"Reviewed in the United States on July 7, 2021","{'neg': 0.046, 'neu': 0.751, 'pos': 0.202, 'co...",0.9944,positive
2,Assault Kittens,best 14inch laptop world,sold m1 macbook one best decisions lifeif youv...,1.0 out of 5 stars,Verified Purchase,"Reviewed in the United States on June 18, 2021","{'neg': 0.0, 'neu': 0.748, 'pos': 0.252, 'comp...",0.8779,positive


In [30]:
def to_lower(text):
    if pd.isnull(text):
        return ""
    return text.lower()

# Apply to both columns
df['Review'] = df['Review'].apply(to_lower)
df['Title'] = df['Title'].apply(to_lower)

In [31]:
def remove_punctuation(text):
    if pd.isnull(text):
        return ""
    return re.sub(r'[^\w\s]', '', text)  # removes anything except letters, numbers, and whitespace

# Apply to both columns
df['Review'] = df['Review'].apply(remove_punctuation)
df['Title'] = df['Title'].apply(remove_punctuation)

In [32]:
def clean_text(text):
    if pd.isnull(text):
        return ""

    # Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", '', text, flags=re.MULTILINE)

    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)

    # Remove Emails
    text = re.sub(r'\S+@\S+', '', text)

    # Remove Digits
    text = re.sub(r'\d+', '', text)

    # Remove Mentions (@usernames)
    text = re.sub(r'@\w+', '', text)

    # Remove Hashtags (#word)
    text = re.sub(r'#\w+', '', text)

    # Remove Emojis (basic Unicode filter)
    emoji_pattern = re.compile(
        "["
        u"\U0001F600-\U0001F64F"  # emoticons
        u"\U0001F300-\U0001F5FF"  # symbols & pictographs
        u"\U0001F680-\U0001F6FF"  # transport & map symbols
        u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
        "]+", flags=re.UNICODE
    )
    text = emoji_pattern.sub(r'', text)

    # Final trim
    text = text.strip()
    return text
    df['Review'] = df['Review'].apply(clean_text)
df['Title'] = df['Title'].apply(clean_text)


In [36]:
stop_words = set(stopwords.words('english'))
nltk.download('wordnet')

# Function to remove stopwords
def remove_stopwords(text):
    if pd.isnull(text):
        return ""
    words = word_tokenize(text)
    filtered_words = [word for word in words if word.lower() not in stop_words]
    return ' '.join(filtered_words)

# Apply to the 'Review' column
df['Review'] = df['Review'].apply(remove_stopwords)
df['Title'] = df['Title'].apply(remove_stopwords)

[nltk_data] Downloading package wordnet to /root/nltk_data...


In [38]:
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk.corpus import wordnet
lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    if pd.isnull(text):
        return ""

    # Tokenize and lemmatize each word
    words = word_tokenize(text)
    lemmatized_words = [lemmatizer.lemmatize(word) for word in words]

    return ' '.join(lemmatized_words)
df['Review'] = df['Review'].apply(lemmatize_text)
df['Title'] = df['Title'].apply(lemmatize_text)


In [39]:
# Remove rows where 'Review' or 'Title' is empty, contains only spaces, or is NaN
df = df[~df['Review'].astype(str).str.strip().eq('')]
df = df[~df['Title'].astype(str).str.strip().eq('')]

# Reset index after dropping
df.reset_index(drop=True, inplace=True)


In [40]:
def feature_rating(text, dataframe) -> pd.DataFrame:
    """
    Extracts numerical ratings from a string column and adds them as a new column to the DataFrame.

    Parameters
    ----------
    text : pd.Series
        Column from which to extract the rating (e.g., "4.5 out of 5").

    dataframe : pd.DataFrame
        Original DataFrame to which the new 'New rating' column will be added.

    Returns
    -------
    pd.DataFrame
        Updated DataFrame with a 'New rating' column of floats.
    """
    rating = []
    for i in range(len(text)):
        try:
            row = str(text[i]).split(" ")[0].replace(",", ".")
            row = float(row)
        except Exception:
            row = None  # or use np.nan
        rating.append(row)

    new_ratings = pd.DataFrame({"New rating": rating})
    dataframe = pd.concat([dataframe, new_ratings["New rating"]], axis=1)

    return dataframe

# Apply to your dataset
df = feature_rating(text=df["Rating"], dataframe=df)

In [41]:
from datetime import datetime

def feature_date(text, dataframe) -> pd.DataFrame:
    """
    Extracts the date from text and converts it to datetime.

    Parameters
    ----------
    text : pd.Series
        Column from which to extract the review's date (e.g., 'Reviewed in the United States on March 5, 2022').

    dataframe : pd.DataFrame
        The original DataFrame to which the 'New date' column will be added.

    Returns
    -------
    pd.DataFrame
        DataFrame with an additional 'New date' column.
    """
    parsed_dates = []

    for entry in text:
        try:
            # Split and get the last 3 words as date
            parts = str(entry).strip().split(" ")
            # Example format: "Reviewed in the United States on March 5, 2022"
            month = parts[-3]
            day = parts[-2].replace(",", "")
            year = parts[-1]

            # Fix typo like "0ctober" → "October"
            if month.lower().startswith("0c"):
                month = "October"

            date_str = f"{month} {day} {year}"
            date_obj = datetime.strptime(date_str, "%B %d %Y")
            parsed_dates.append(date_obj)
        except Exception:
            parsed_dates.append(pd.NaT)  # Not a Time for invalid entries

    # Add to DataFrame
    new_date_col = pd.DataFrame({"New date": parsed_dates})
    dataframe = pd.concat([dataframe, new_date_col], axis=1)

    return dataframe

# Apply to your DataFrame
df = feature_date(text=df["Date"], dataframe=df)

In [42]:
df["Verified Purchase"].replace(
    to_replace="Verified Purchase",
    value="Yes",
    inplace=True
)

In [43]:
def feature_country(text, dataframe) -> pd.DataFrame:
    """
    Extracts the country from the review date string (e.g., 'Reviewed in Canada on March 5, 2022').

    Parameters
    ----------
    text : pd.Series
        Column from which to extract the country.

    dataframe : pd.DataFrame
        Original DataFrame to update.

    Returns
    -------
    pd.DataFrame
        Updated DataFrame with a new 'Country' column.
    """
    countries = []

    for entry in text:
        try:
            parts = str(entry).split(" ")
            # Expecting format: "Reviewed in <Country> on <Date>"
            if "Reviewed" in parts and "in" in parts:
                idx = parts.index("in")
                country = parts[idx + 1]
                # Handle "United States" or multi-word countries
                if idx + 2 < len(parts) and parts[idx + 2].lower() != "on":
                    country += " " + parts[idx + 2]
            else:
                country = "Unknown"
        except:
            country = "Unknown"
        countries.append(country)

    country_df = pd.DataFrame({"Country": countries})
    dataframe = pd.concat([dataframe, country_df["Country"]], axis=1)

    return dataframe

# Apply to your DataFrame
df = feature_country(text=df["Date"], dataframe=df)


In [44]:
df.to_csv(path_or_buf="/content/amzn_customer_reviews.csv", sep=",", index=False)
